# Pipeline NILM — experimentación en Colab

Notebook delgado: toda la lógica vive en el paquete `nilm_pipeline/` (repo `NILM_pipeline`).
Cada experimento es una instancia de `Config`; los resultados se acumulan en `outputs/leaderboard.csv`.

**Fuentes (GitHub, repos privados)**:
- código: `github.com/klopszteinleonel/NILM_pipeline`
- datos:  `github.com/klopszteinleonel/NILM_local_dataset` (una carpeta por clase, `combinadas/`, `nada/`)

**Antes de correr en Colab**: crear un *secret* llamado `GITHUB_TOKEN` (ícono de la llave, barra lateral)
con un token de GitHub con permiso de lectura sobre los dos repos, y habilitarlo para este notebook.
La celda 0 clona ambos repos en `/content/` (o hace `git pull` si ya están) e instala dependencias.
`outputs/` en Colab es efímero: para conservar el leaderboard entre sesiones poné `PERSISTIR_EN_DRIVE = True`.

Principio rector: todo lo que hace el pipeline tiene que poder correr en el ESP32-S3 (ver `AGENTS.md`).

In [ ]:
# 0. Setup: en Colab clona los repos de GitHub; en local usa rutas relativas
import os, sys, subprocess

REPO_PIPELINE = "klopszteinleonel/NILM_pipeline"
REPO_DATOS    = "klopszteinleonel/NILM_local_dataset"
PERSISTIR_EN_DRIVE = False     # True: outputs/ en MyDrive/NILM/outputs (sobrevive a la sesión)

def clonar_o_actualizar(repo, destino, token):
    url = f"https://{'x-access-token:' + token + '@' if token else ''}github.com/{repo}.git"
    if os.path.isdir(os.path.join(destino, ".git")):
        subprocess.run(["git", "-C", destino, "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, destino], check=True)
    print(f"{repo} -> {destino}")

try:
    from google.colab import userdata
    try:
        TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        TOKEN = ""          # sin secret: solo funciona si los repos son públicos
        print("[AVISO] No hay secret GITHUB_TOKEN; se intenta clonar sin token")
    RUTA_PIPELINE = "/content/NILM_pipeline"
    RUTA_DATOS    = "/content/NILM_local_dataset"
    clonar_o_actualizar(REPO_PIPELINE, RUTA_PIPELINE, TOKEN)
    clonar_o_actualizar(REPO_DATOS, RUTA_DATOS, TOKEN)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{RUTA_PIPELINE}/requirements-colab.txt"], check=True)
    if PERSISTIR_EN_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUTA_SALIDAS = "/content/drive/MyDrive/NILM/outputs"
    else:
        RUTA_SALIDAS = os.path.join(RUTA_PIPELINE, "outputs")
    EN_COLAB = True
except ImportError:
    RUTA_PIPELINE = os.path.abspath(os.path.join(os.getcwd(), ".."))
    RUTA_DATOS    = os.path.abspath(os.path.join(RUTA_PIPELINE, "..", "clienteTCP", "capturas_limpio"))
    RUTA_SALIDAS  = os.path.join(RUTA_PIPELINE, "outputs")
    EN_COLAB = False
os.makedirs(RUTA_SALIDAS, exist_ok=True)
sys.path.insert(0, RUTA_PIPELINE)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import nilm_pipeline
from nilm_pipeline import *
from nilm_pipeline import plots
print(f"nilm_pipeline {nilm_pipeline.__version__} | colab={EN_COLAB}\ndatos:   {RUTA_DATOS}\nsalidas: {RUTA_SALIDAS}")

## 1. Ingesta
Escanea la carpeta de capturas, arma el manifest (con hash del dataset) y una cache `.npz`.
Para agregar datos: copiar los JSON a la carpeta que corresponda y volver a correr esta celda.

In [ ]:
ds = cargar_dataset(RUTA_DATOS)
print(ds.resumen())
guardar_manifest(ds.manifest_filas, os.path.join(RUTA_SALIDAS, "manifest.csv"))
guardar_cache(ds, os.path.join(RUTA_SALIDAS, "cache_dataset.npz"))
fs = ds.fs()
sen, Y, G = ds.senales(), ds.y(), ds.grupos()

## 2. Configuración base
`Config` es la única fuente de verdad. El baseline es edge-friendly (OvR de RandomForest 20×8 ≈ 276 KB en C).

In [ ]:
CFG = Config()
print(CFG.nombre())
pd.Series(CFG.to_dict()).to_frame("valor").T

## 3. Sanity checks visuales
Señal, detalle y espectro (con las armónicas de la f0 estimada) de una captura por clase; una mezcla sintética; augment.

In [ ]:
for clase in ("aire", "tele", "calefon"):
    r = next((r for r in ds.registros if r.clases == (clase,)), None)
    if r is not None:
        plots.plot_captura(r.datos, fs, CFG, titulo=f"{clase} — {r.ruta}")
plt.show()

In [ ]:
rng = np.random.default_rng(CFG.seed)
m = sintetizar_combinadas(sen, Y, G, ds.clases, CFG, rng, total=3)
i = 0
origen = [j for j in range(len(G)) if G[j] in m.origenes[i]]
print("mezcla 0 =", [c for c, b in zip(ds.clases, m.Y[i]) if b], "| fuentes:", m.origenes[i])
plots.plot_mix([sen[j] for j in origen], [G[j].split('/')[-1][:24] for j in origen], m.senales[i], fs)
V = segmentar(sen[0], CFG.W, CFG.solape)
Vp, esc = preprocesar_ventanas(V, CFG)
plots.plot_augment(Vp[0], CFG, fs)
plt.show()

## 4. Features
Mediana por clase de algunas features `compact`. La amplitud absoluta (`rms_A`, `h1_A`) es lo único que separa a las cargas resistivas (calefon/plancha/horno/secador).

In [ ]:
X, Yw, Gw, idx = segmentar_dataset(sen, Y, G, CFG)
Vp, esc = preprocesar_ventanas(X, CFG)
F, nombres = extraer_features(Vp, esc, CFG, fs)
df_f = pd.DataFrame(F, columns=nombres)
df_f["clase"] = [",".join(c for c, b in zip(ds.clases, y) if b) or "nada" for y in Yw]
print(f"{len(F)} ventanas x {len(nombres)} features")
df_f.groupby("clase")[["rms_A", "h1_A", "thd", "h3_rel", "par_impar", "centroide_hz", "cresta"]].median().round(3)

## 5. Baseline
`run_experiment` hace la validación cruzada por captura con mezcla dentro del fold y reporta tres bloques:
`ind` (individuales de test), `mix` (mezclas sintéticas generadas solo con capturas de test) y `real` (combinadas reales, cuando existan).
Métricas por ventana (`v`) y por captura (`c`, promediando las probabilidades de sus ventanas).

In [ ]:
LEADERBOARD = os.path.join(RUTA_SALIDAS, "leaderboard.csv")
res = run_experiment(ds, CFG, leaderboard=LEADERBOARD)
print(reporte(res.metricas, res.clases))

## 6. Barridos
Editá el diccionario `variantes` (cada entrada es un `CFG.clonar(...)`). Todo va al leaderboard; el gráfico muestra F1 vs costo del modelo (nodos → bytes en flash).

In [ ]:
variantes = {
    "W1024":       CFG.clonar(W=1024),
    "W4096":       CFG.clonar(W=4096),
    "blackman":    CFG.clonar(ventana="blackman"),
    "rawfft":      CFG.clonar(estrategia="rawfft"),
    "hp20_causal": CFG.clonar(filtro="hp", causal=True),
    "sin_aug":     CFG.clonar(aug_n=0),
    "aug8":        CFG.clonar(aug_n=8),
    "rf50x10":     CFG.clonar(n_estimators=50, max_depth=10),
    "et":          CFG.clonar(modelo="et"),
}
for nombre, c in variantes.items():
    r = run_experiment(ds, c, leaderboard=LEADERBOARD, verbose=False)
    print(f"{nombre:12s} ind_c_f1_micro={r.metricas['ind_c_f1_micro']:.3f}  "
          f"mix_c_f1_micro={r.metricas.get('mix_c_f1_micro', float('nan')):.3f}  "
          f"nodos={r.metricas['nodos_totales']:7d}  {r.metricas['segundos']:.0f} s")

In [ ]:
lb = cargar_leaderboard(LEADERBOARD, ordenar_por="mix_c_f1_micro")
cols = ["nombre", "ind_c_f1_micro", "ind_c_f1_macro", "mix_c_f1_micro", "mix_c_f1_macro", "nodos_totales", "segundos"]
display(lb[[c for c in cols if c in lb.columns]].head(15))
plots.plot_leaderboard(lb)
plt.show()

## 7. Evaluación del mejor
Elegí `MEJOR` a mano (por F1 **y** por `nodos_totales`) o dejá la primera fila del leaderboard.
LOGO (leave-one-capture-out) es la evaluación definitiva de individuales (61 entrenamientos: ~2 min con `compact`, ~15 min con `rawfft`); las matrices por clase y los umbrales óptimos salen de las predicciones out-of-fold.

In [ ]:
MEJOR = Config.from_json(lb.iloc[0]["config"])
print("MEJOR:", MEJOR.nombre())
res_m = run_experiment(ds, MEJOR, verbose=False)
print(reporte(res_m.metricas, res_m.clases))
res_logo = run_experiment(ds, MEJOR.clonar(cv="logo"), verbose=False)
print("\nLOGO individuales por captura:",
      {k: round(res_logo.metricas[f"ind_c_{k}"], 3) for k in ("exact_match", "f1_micro", "f1_macro")})

In [ ]:
b = res_m.bloques["ind"]
Pg, Yg, _ = agregar_por_grupo(b["P"], b["Y"], b["grupos"])
umbrales = umbrales_optimos(Yg, Pg)
print("umbrales por clase:", dict(zip(res_m.clases, np.round(umbrales, 2).tolist())))
print("F1 macro 0.5 ->", round(metricas_multilabel(Yg, Pg, 0.5)["f1_macro"], 3),
      "| con umbrales:", round(metricas_multilabel(Yg, Pg, umbrales)["f1_macro"], 3))
plots.plot_matrices_por_clase(Yg, Pg, umbrales, res_m.clases)
plots.plot_importancias(res_m.importancias)
plt.show()

## 8. Export al ESP32-S3 y test de paridad
Entrena con todas las individuales (+ mezclas + augment), genera `nilm_modelo.h`, `nilm_config.h`, `pipeline_config.json` y `modelo.joblib`, y compila un programa C en el host que debe reproducir `predict_proba` de sklearn (diferencia < 1e-5).

In [ ]:
modelo, nombres_f, clases = entrenar_final(ds, MEJOR)
CARPETA_EXPORT = os.path.join(RUTA_SALIDAS, f"export_{MEJOR.nombre()}")
info = exportar(modelo, MEJOR, fs, nombres_f, clases, CARPETA_EXPORT, umbrales=umbrales)
print(info)
F_all, _, _, _ = armar_conjunto(sen, Y, G, MEJOR, fs)
paridad = test_paridad(CARPETA_EXPORT, F_all, predecir_proba(modelo, F_all, len(clases)))
print(paridad)
assert paridad["paridad_ok"], "el C exportado NO reproduce a sklearn: no flashear"
rob = robustez_snr(modelo, ds, MEJOR)
print("robustez SNR:", {k: round(v, 3) for k, v in rob.items()})
plots.plot_robustez(rob)
plt.show()
print("archivos en", CARPETA_EXPORT, ":", sorted(os.listdir(CARPETA_EXPORT)))

## 9. (Opcional) Modelos de referencia
No exportables al borde (o muy lentos); sirven para saber cuánto margen hay. `hgb` tarda varios minutos.

In [ ]:
CORRER_REFERENCIA = False
if CORRER_REFERENCIA:
    for m in ("knn", "logreg", "svm", "hgb"):
        r = run_experiment(ds, MEJOR.clonar(modelo=m), leaderboard=LEADERBOARD, verbose=False)
        print(f"{m:7s} ind_c_f1_micro={r.metricas['ind_c_f1_micro']:.3f} "
              f"mix_c_f1_micro={r.metricas.get('mix_c_f1_micro', float('nan')):.3f} {r.metricas['segundos']:.0f} s")